# Week 5 Day 1 — XGBoost

## What I built:
- XGBoost classifier on Titanic — 84.29% CV accuracy
- XGBoost regressor on House Prices — 0.1287 CV RMSE
- Compared XGBoost vs Random Forest on both datasets

## Key XGBoost parameters:
- learning_rate (eta) = 0.05 → small steps, more trees needed
- max_depth = 4 → shallow trees prevent overfitting
- subsample = 0.8 → use 80% rows per tree (row sampling)
- colsample_bytree = 0.8 → use 80% features per tree
- eval_metric = 'rmse'/'logloss' → internal training metric
- verbosity = 0 → silent output

## XGBoost vs Random Forest:
Random Forest → parallel trees, averaging, bootstrap sampling
XGBoost      → sequential trees, each fixes previous errors,
               learning rate controls step size

## Results:
Titanic  : XGB 0.8429 vs RF 0.8418 → XGB wins (+0.0011)
House Prices: XGB 0.1287 vs RF 0.1315 → XGB wins (-0.0028)

## Kaggle score: 0.12930 ← new best (was 0.14864)

# Install and first XGBoost model on Titanic:

In [2]:
pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/48.9 MB 1.4 MB/s eta 0:00:35
    --------------------------------------- 0.8/48.9 MB 1.5 MB/s eta 0:00:32
    --------------------------------------- 1.0/48.9 MB 1.5 MB/s eta 0:00:33
   - -------------------------------------- 1.6/48.9 MB 1.4 MB/s eta 0:00:33
   - -------------------------------------- 1.6/48.9 MB 1.4 MB/s eta 0:00:33
   - -------------------------------------- 2.1/48.9 MB 1.4 MB/s eta 0:00:33
   - -------------------------------------- 2.4/48.9 MB 1.4 MB/s eta 0:00:33
   -- ------------------------------------- 2.6/48.9 MB 1.4 MB/s eta 0:00:33
   -- ------------------------------------- 2.9/48.9 MB 1.4 MB/s eta 0:00:33
   -- ------------------------------------- 3.1/48.9 MB 1.4 MB/s eta 0:00:33
   -- --------------

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier, XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
import warnings
warnings.filterwarnings('ignore')

# load Titanic
df = pd.read_csv('train.csv')
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone']    = (df['FamilySize'] == 1).astype(int)
df['Title']      = df['Name'].apply(
                       lambda name: name.split(',')[1]
                                        .split('.')[0].strip())
df['Title']      = df['Title'].map({
                       'Mr': 0, 'Miss': 1, 'Mrs': 2,
                       'Master': 3}).fillna(4)

num_cols = ['Age', 'Fare', 'FamilySize', 'Title']
cat_cols = ['Sex', 'Embarked', 'Pclass']

X = df[num_cols + cat_cols]
y = df['Survived']

# preprocessor — use OrdinalEncoder for XGBoost
# XGBoost handles ordinal encoded cats better than OHE
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value',
                               unknown_value=-1))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# XGBoost pipeline
xgb_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss',
        verbosity=0
    ))
])

# evaluate
skfold = StratifiedKFold(n_splits=5)
scores = cross_val_score(xgb_pipe, X, y,
                         cv=skfold, scoring='accuracy')

print("=" * 50)
print("   XGBOOST — TITANIC")
print("=" * 50)
print(f"  CV Accuracy : {scores.mean():.4f} ± {scores.std():.4f}")
print(f"  RF baseline : 0.8418")
print(f"  Improvement : {scores.mean() - 0.8418:+.4f}")
print("=" * 50)

   XGBOOST — TITANIC
  CV Accuracy : 0.8429 ± 0.0219
  RF baseline : 0.8418
  Improvement : +0.0011


# XGBoost on House Prices:

In [5]:
# load House Prices
train = pd.read_csv('house_prices_train.csv')
test  = pd.read_csv('house_prices_test.csv')

# feature engineering
def engineer_features(df):
    df = df.copy()
    df['TotalSF']     = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath']   = (df['FullBath'] + 0.5*df['HalfBath'] +
                         df['BsmtFullBath'] + 0.5*df['BsmtHalfBath'])
    df['HouseAge']    = df['YrSold'] - df['YearBuilt']
    df['RemodAge']    = df['YrSold'] - df['YearRemodAdd']
    df['HasPool']     = (df['PoolArea'] > 0).astype(int)
    df['HasGarage']   = (df['GarageArea'] > 0).astype(int)
    df['HasBasement'] = (df['TotalBsmtSF'] > 0).astype(int)
    return df

train_fe = engineer_features(train)
test_fe  = engineer_features(test)

num_cols_hp = train_fe.select_dtypes(
    include=['int64','float64']).columns.tolist()
num_cols_hp = [c for c in num_cols_hp if c != 'SalePrice']
cat_cols_hp = train_fe.select_dtypes(include=['object']).columns.tolist()

X_hp = train_fe.drop(columns=['SalePrice', 'Id'])
y_hp = np.log1p(train['SalePrice'])

# update col lists
num_cols_hp = X_hp.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols_hp = X_hp.select_dtypes(include=['object']).columns.tolist()

# preprocessor for House Prices
num_tr_hp = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_tr_hp = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value',
                               unknown_value=-1))
])
preprocessor_hp = ColumnTransformer(transformers=[
    ('num', num_tr_hp, num_cols_hp),
    ('cat', cat_tr_hp, cat_cols_hp)
])

# XGBoost regressor
xgb_reg_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor_hp),
    ('model', XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='rmse',
        verbosity=0
    ))
])

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_xgb = cross_val_score(xgb_reg_pipe, X_hp, y_hp,
                              cv=kfold,
                              scoring='neg_root_mean_squared_error')

print("=" * 50)
print("   XGBOOST — HOUSE PRICES")
print("=" * 50)
print(f"  XGBoost RMSE : {-scores_xgb.mean():.4f} ± {scores_xgb.std():.4f}")
print(f"  RF baseline  : 0.1315")
print(f"  Improvement  : {(-scores_xgb.mean() - 0.1315):+.4f}")
print("=" * 50)

   XGBOOST — HOUSE PRICES
  XGBoost RMSE : 0.1287 ± 0.0185
  RF baseline  : 0.1315
  Improvement  : -0.0028


# Submit to Kaggle:

In [6]:
# fit on full training data
xgb_reg_pipe.fit(X_hp, y_hp)

# predict on test
X_test_hp = test_fe.drop(columns=['Id'])
log_preds = xgb_reg_pipe.predict(X_test_hp)
predictions = np.expm1(log_preds)

submission = pd.DataFrame({
    'Id'       : test['Id'],
    'SalePrice': predictions
})
submission.to_csv('xgb_submission_v1.csv', index=False)
print("Submission saved!")

Submission saved!
